In [1]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
## 데이터셋, 데이터로더 관련 모듈

from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk
from nltk.tokenize import word_tokenize

In [2]:
FILE_DIR = './data/'
FILE_PATH = FILE_DIR+'IMDb_Reviews.csv'


In [3]:
import string


STOPWORDS = stopwords.words('english')
TOKENIZER = get_tokenizer('basic_english')
PUNC = string.punctuation
PUNC = PUNC+''.join([str(x) for x in range(10)])

In [4]:
def yield_tokens(data):
    for line in data:
        line = ''.join([x for x in line if x not in PUNC])
        yield word_tokenize(line.lower())

In [5]:
# nltk.download('punkt_tab')

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    data = f.readlines()
    tokens = yield_tokens(data)

VOCAB = build_vocab_from_iterator(tokens, specials=["<unk>", "<pad>"])
VOCAB.set_default_index(VOCAB["<unk>"])

In [6]:
### tokens를 정수화++ 길이맞추기 해야함

In [7]:
def encode_texts(data, vocab):
    encoded = []
    for line in data:
        token_ids = [vocab[token] for token in tokens]
        encoded.append(torch.tensor(token_ids, dtype=torch.long))
    return encoded

In [9]:
encoded_sequences = encode_texts(data, VOCAB)
padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=VOCAB["<pad>"])

print(padded_sequences.shape) 
print(padded_sequences[0])   

torch.Size([50001, 0])
tensor([], dtype=torch.int64)


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
##DS만들기

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# 1. 데이터 분할 (선택 사항)
train_padded, temp_padded, train_labels, temp_labels = train_test_split(padded_sequences, labels, test_size=0.3, random_state=42)
val_padded, test_padded, val_labels, test_labels = train_test_split(temp_padded, temp_labels, test_size=0.5, random_state=42)

# NumPy 배열을 PyTorch Tensor로 변환
train_padded = torch.tensor(train_padded, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.long)
val_padded = torch.tensor(val_padded, dtype=torch.long)
val_labels = torch.tensor(val_labels, dtype=torch.long)
test_padded = torch.tensor(test_padded, dtype=torch.long)
test_labels = torch.tensor(test_labels, dtype=torch.long)

# 2. 커스텀 데이터셋 클래스 정의
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        encoding = self.encodings[idx]
        label = self.labels[idx]
        return encoding, label

train_dataset = TextDataset(train_padded, train_labels)
val_dataset = TextDataset(val_padded, val_labels)
test_dataset = TextDataset(test_padded, test_labels)